# 06 - Charts and Summary

Turn a query result into what the user sees:

1. **Chart:** picked by **rules** from the result's shape, not by the
   model, so the same result always gets the same chart. A single number
   becomes a stat tile, a ranking becomes horizontal bars in one colour,
   a time series becomes a line, two measures on different scales become
   small multiples (never two y-axes). Anything else stays a table.
2. **Summary:** one to three plain sentences from the model, then a
   **number check**: every number in the summary must come from the
   result (or the question). One retry, then a fixed template.
3. **Caveats:** added by code, not the model, from the cleaning flags of
   the columns the query used (missing values, extreme values, similar
   category spellings).

The colours are the validated reference palette (light and dark
selected separately); the table view always sits beside the chart.

Becomes `src/helpers/chart_builder.py` (and the summary step in
`pipeline.py`).

In [1]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

import json
import logging
import math
import re
import sqlite3
import statistics
import time
from pathlib import Path

import httpx
import ollama
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


BASE_DIR = (
    Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
)
DATASETS_DIR = BASE_DIR / "data" / "output_data" / "datasets"
MODEL_PROFILE_PATH = BASE_DIR / "data" / "output_data" / "model" / "model_profile.json"
DEV_RESULTS_PATH = BASE_DIR / "data" / "output_data" / "self_correction" / "dev_results.csv"
OUTPUT_DIR = BASE_DIR / "data" / "output_data" / "charts"

OLLAMA_URL = "http://localhost:11434"
LLM_TEMPERATURE = 0
KEEP_ALIVE = "30m"

# ---- chart rules ---------------------------------------------
MAX_BARS = 25              # more categories -> top 25 + note
MAX_SERIES = 8             # categorical palette size; more -> table
MAX_SMALL_MULTIPLES = 3    # measures shown as separate panels
MIN_SCATTER_ROWS = 5
MIN_HISTOGRAM_ROWS = 10
BAR_SLOT_PX = 36           # bar ~22 px thick with BAR_GAP 0.4 (cap 24 px)
BAR_GAP = 0.4
CHART_WIDTH = 760

# ---- summary -------------------------------------------------
SUMMARY_MAX_ROWS = 20      # rows of the result shown to the model
ALWAYS_ALLOWED_NUMBERS = set(range(0, 13))   # "3 cities", "top 5"
NUMBER_PATTERN = re.compile(r"(?<![\w.])-?\d+(?:,\d{3})*(?:\.\d+)?")

# ---- palette (validated reference instance, see project notes) ----
THEMES = {
    "light": {
        "surface": "#fcfcfb", "ink": "#0b0b0b", "ink_secondary": "#52514e",
        "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7",
        "series": ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                   "#e87ba4", "#008300", "#6250d6", "#e34948"],
    },
    "dark": {
        "surface": "#1a1a19", "ink": "#f0efec", "ink_secondary": "#c3c2b7",
        "muted": "#898781", "grid": "#2c2c2a", "axis": "#383835",
        "series": ["#3987e5", "#d95926", "#199e70", "#c98500",
                   "#d55181", "#008300", "#9085e9", "#e66767"],
    },
}
FONT_FAMILY = 'system-ui, -apple-system, "Segoe UI", sans-serif'

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger("06_charts_summary")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with MODEL_PROFILE_PATH.open(encoding="utf-8") as file:
    model_settings = json.load(file)["selected"]

print("Model:", model_settings["model"])

Model: qwen2.5:7b-instruct


## 2. Code from notebook 04

Model call, SQLite statistics functions and the read-only query runner,
copied unchanged.

In [2]:
# ============================================================
# 2. FROM NOTEBOOK 04: MODEL CALL, STATISTICS, QUERY RUNNER
# ============================================================

# ---- notebook 03: model call --------------------------------

# Cap on generated tokens: a runaway answer becomes invalid JSON (and is
# retried) instead of running into the request timeout.
MAX_OUTPUT_TOKENS = 512


def get_client(timeout):
    return ollama.Client(host=OLLAMA_URL, timeout=timeout)


def chat_json(settings, messages, schema):
    """Call the model with a JSON schema; return (answer, stats)."""

    client = get_client(settings["request_timeout"])
    started = time.perf_counter()
    response = client.chat(
        model=settings["model"], messages=messages, format=schema,
        options={"temperature": LLM_TEMPERATURE,
                 "num_ctx": settings["num_ctx"],
                 "num_predict": MAX_OUTPUT_TOKENS},
        keep_alive=KEEP_ALIVE,
    )
    content = response["message"]["content"]

    try:
        answer = json.loads(content)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Model did not return valid JSON: {content[:200]!r}") from exc

    missing = [key for key in schema.get("required", []) if key not in answer]

    if missing:
        raise ValueError(f"Model answer is missing keys {missing}: {answer}")

    stats = {"seconds": round(time.perf_counter() - started, 2),
             "prompt_tokens": response["prompt_eval_count"],
             "output_tokens": response["eval_count"]}

    return answer, stats


# ---- notebook 02: statistics SQLite lacks -------------------

def json_numbers(json_array):
    try:
        values = json.loads(json_array) if json_array else []
    except (TypeError, ValueError):
        return []

    numbers = []

    for value in values if isinstance(values, list) else []:
        if isinstance(value, (int, float)) and not isinstance(value, bool) \
                and not math.isnan(value):
            numbers.append(float(value))

    return numbers


def median_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.median(numbers) if numbers else None


def stdev_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.stdev(numbers) if len(numbers) > 1 else None


def variance_of(json_array):
    numbers = json_numbers(json_array)
    return statistics.variance(numbers) if len(numbers) > 1 else None


def percentile_of(json_array, percent):
    numbers = sorted(json_numbers(json_array))

    if not numbers or percent is None or not 0 <= percent <= 100:
        return None

    position = (len(numbers) - 1) * percent / 100
    lower, upper = math.floor(position), math.ceil(position)

    return numbers[lower] + (numbers[upper] - numbers[lower]) * (position - lower)


SQL_FUNCTIONS = {"median_of": (median_of, 1), "stdev_of": (stdev_of, 1),
                 "variance_of": (variance_of, 1),
                 "percentile_of": (percentile_of, 2)}


def register_functions(connection):
    for name, (function, arg_count) in SQL_FUNCTIONS.items():
        connection.create_function(name, arg_count, function,
                                   deterministic=True)
    return connection


# ---- running a query (full guardrails come in notebook 05) --

def run_query(database_path, sql):
    """Run SQL on a read-only connection and return a DataFrame."""

    uri = Path(database_path).resolve().as_uri() + "?mode=ro"

    with sqlite3.connect(uri, uri=True) as connection:
        register_functions(connection)
        return pd.read_sql_query(sql, connection)


def load_schema_text(dataset_name):
    return (DATASETS_DIR / dataset_name / "schema.txt").read_text(encoding="utf-8")


def database_path(dataset_name):
    return DATASETS_DIR / dataset_name / "dataset.db"

## 3. Reading the result's shape

Each column is **numeric**, **temporal** (dates, months like `2024-01`,
years) or **category** (any other text). The chart is chosen from these
counts.

In [3]:
# ============================================================
# 3. READING THE RESULT'S SHAPE
# ============================================================

TEMPORAL_VALUE = re.compile(r"^\d{4}(-\d{2}(-\d{2}( \d{2}:\d{2}(:\d{2})?)?)?)?$")
TEMPORAL_NAME = re.compile(r"(date|month|year|week|day|quarter|period|time)",
                           re.IGNORECASE)


def is_temporal(series):
    """ISO-like dates / months, or a year column named like one."""

    values = series.dropna().astype(str)

    if values.empty or not values.str.match(TEMPORAL_VALUE).all():
        return False

    # Plain 4-digit numbers are only years when the name says so.
    if values.str.fullmatch(r"\d{4}").all():
        return bool(TEMPORAL_NAME.search(str(series.name)))

    return True


def column_roles(frame):
    """{"temporal": [...], "numeric": [...], "category": [...]}"""

    roles = {"temporal": [], "numeric": [], "category": []}

    for column in frame.columns:
        series = frame[column]

        if is_temporal(series):
            roles["temporal"].append(column)
        elif pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series):
            roles["numeric"].append(column)
        else:
            roles["category"].append(column)

    return roles


example = pd.DataFrame({"month": ["2024-01", "2024-02"], "sales": [10, 20],
                        "city": ["Delhi", "Pune"], "year": [2024, 2024]})
column_roles(example)

{'temporal': ['month', 'year'], 'numeric': ['sales'], 'category': ['city']}

## 4. Choosing the chart

| Result shape | Chart |
|---|---|
| 1 row, 1 number | **stat tile** |
| time + 1 number | line |
| time + 2-3 numbers | small multiples (one panel per measure, own y-axis) |
| time + category (<= 8) + number | multi-line, one colour per category |
| category + number | horizontal bars, largest first, one colour |
| 2 categories (second <= 8) + number | grouped horizontal bars |
| 2 numbers, no category | scatter |
| 1 number, many rows | histogram |
| anything else | table only |

In [4]:
# ============================================================
# 4. CHOOSING THE CHART
# ============================================================

def choose_chart(frame, question=""):
    """
    Pick a chart from the result's shape.

    Returns:
        spec dict: {"type", "x", "y", "color", "title", "note"}
        type: stat | line | multiples | bar | grouped_bar | scatter |
              histogram | table | none
    """

    spec = {"type": "table", "x": None, "y": None, "color": None,
            "title": question.strip().rstrip("?") or None, "note": None}

    if frame is None or frame.empty:
        return {**spec, "type": "none"}

    roles = column_roles(frame)
    temporal, numeric, category = roles["temporal"], roles["numeric"], roles["category"]
    rows = len(frame)

    # Single headline number.
    if rows == 1 and len(numeric) == 1:
        return {**spec, "type": "stat", "y": numeric[0]}

    # Change over time.
    if temporal and numeric:
        x = temporal[0]

        if category and frame[category[0]].nunique() <= MAX_SERIES:
            return {**spec, "type": "line", "x": x, "y": numeric[0],
                    "color": category[0]}

        if category:
            return {**spec, "note": f"Too many {category[0]} values to chart."}

        if len(numeric) == 1:
            return {**spec, "type": "line", "x": x, "y": numeric[0]}

        shown = numeric[:MAX_SMALL_MULTIPLES]
        note = (f"Showing {len(shown)} of {len(numeric)} measures."
                if len(numeric) > len(shown) else None)
        return {**spec, "type": "multiples", "x": x, "y": shown, "note": note}

    # Magnitude by category.
    if category and numeric:
        if len(category) >= 2:
            second = category[1]

            if frame[second].nunique() <= MAX_SERIES:
                return {**spec, "type": "grouped_bar", "x": numeric[0],
                        "y": category[0], "color": second}

            return {**spec, "note": f"Too many {second} values to chart."}

        note = None

        if len(numeric) > 1:
            note = f"Chart shows {numeric[0]}; all columns are in the table."

        if rows > MAX_BARS:
            note = f"Chart shows the top {MAX_BARS} of {rows} rows."

        return {**spec, "type": "bar", "x": numeric[0], "y": category[0],
                "note": note}

    # Two measures against each other.
    if len(numeric) >= 2 and rows >= MIN_SCATTER_ROWS:
        return {**spec, "type": "scatter", "x": numeric[0], "y": numeric[1]}

    # Distribution of one measure.
    if len(numeric) == 1 and rows >= MIN_HISTOGRAM_ROWS:
        return {**spec, "type": "histogram", "x": numeric[0]}

    return spec


for frame in [
    pd.DataFrame({"orders": [32]}),
    pd.DataFrame({"month": ["2024-01", "2024-02"], "sales": [10, 20]}),
    pd.DataFrame({"city": ["A", "B"], "sales": [1, 2]}),
    pd.DataFrame({"region": ["N", "S", "N"], "paid": ["yes", "no", "no"], "n": [1, 2, 3]}),
    pd.DataFrame({"a": range(6), "b": range(6)}),
    pd.DataFrame({"name": ["x", "y"]}),
]:
    print(f"{choose_chart(frame)['type']:<12} {list(frame.columns)}")

stat         ['orders']
line         ['month', 'sales']
bar          ['city', 'sales']
grouped_bar  ['region', 'paid', 'n']
scatter      ['a', 'b']
table        ['name']


## 5. Drawing the chart

Mark specs from the chart guide: bars at most 24 px thick with rounded
data ends and value labels at the tip; 2 px lines with 8 px markers and a
surface-coloured ring; hairline grid; muted axis text; text never in the
series colour; hover tooltips on every mark. Light and dark themes use
their own selected palette steps.

In [5]:
# ============================================================
# 5. DRAWING THE CHART
# ============================================================

def format_number(value):
    """Labels: thousands separators, at most 2 decimals."""

    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""

    value = float(value)

    return f"{value:,.0f}" if value.is_integer() else f"{value:,.2f}"


def readable(name):
    """total_sales -> Total sales"""

    return str(name).replace("_", " ").strip().capitalize()


def tick_format(values):
    """Compact ticks (10k) once an axis reaches 10,000."""

    numbers = pd.to_numeric(pd.Series(values), errors="coerce").abs()
    return "~s" if numbers.max() >= 10_000 else ",~r"


def base_layout(figure, theme, title, height):
    """Surface, ink, hairline grid and hover for every chart."""

    figure.update_layout(
        title={"text": title, "x": 0, "xanchor": "left",
               "font": {"size": 16, "color": theme["ink"]}},
        width=CHART_WIDTH, height=height,
        margin={"l": 16, "r": 48, "t": 84, "b": 40},
        paper_bgcolor=theme["surface"], plot_bgcolor=theme["surface"],
        font={"family": FONT_FAMILY, "size": 12, "color": theme["ink_secondary"]},
        hoverlabel={"bgcolor": theme["surface"], "bordercolor": theme["axis"],
                    "font": {"color": theme["ink"], "family": FONT_FAMILY}},
        # Legend in one row between the title and the plot, never
        # competing with the x-axis title.
        legend={"orientation": "h", "y": 1.02, "yanchor": "bottom", "x": 0,
                "font": {"color": theme["ink_secondary"]}},
        barcornerradius=4,
    )
    figure.update_xaxes(gridcolor=theme["grid"], gridwidth=1,
                        linecolor=theme["axis"], zeroline=False,
                        tickfont={"color": theme["muted"]}, automargin=True,
                        title_font={"color": theme["ink_secondary"]})
    figure.update_yaxes(gridcolor=theme["grid"], gridwidth=1,
                        linecolor=theme["axis"], zeroline=False,
                        tickfont={"color": theme["muted"]}, automargin=True,
                        title_font={"color": theme["ink_secondary"]})

    return figure


def series_colors(values, theme):
    """Fixed colour per category name (sorted), never by rank."""

    names = sorted(pd.Series(values).dropna().astype(str).unique())
    return {name: theme["series"][index] for index, name in enumerate(names)}


def draw_stat(frame, spec, theme):
    """Hero number; a text column in the row names what it is about."""

    value = frame[spec["y"]].iloc[0]
    labels = [str(frame[column].iloc[0]) for column in frame.columns
              if column != spec["y"] and pd.notna(frame[column].iloc[0])]
    label = " · ".join(labels + [readable(spec["y"])])

    figure = go.Figure(go.Indicator(
        mode="number", value=None if pd.isna(value) else float(value),
        number={"valueformat": ",.2~f",
                "font": {"size": 56, "color": theme["ink"]}},
        title={"text": label,
               "font": {"size": 14, "color": theme["ink_secondary"]}},
    ))
    return base_layout(figure, theme, spec["title"], 220)


def draw_bar(frame, spec, theme):
    data = (frame.dropna(subset=[spec["x"]])
                 .sort_values(spec["x"], ascending=False)
                 .head(MAX_BARS))
    figure = go.Figure(go.Bar(
        x=data[spec["x"]], y=data[spec["y"]].astype(str), orientation="h",
        marker={"color": theme["series"][0], "line": {"width": 0}},
        text=[format_number(value) for value in data[spec["x"]]],
        textposition="outside", cliponaxis=False,
        textfont={"color": theme["ink_secondary"]},
        hovertemplate="%{y}: %{x:,}<extra></extra>",
    ))
    figure.update_layout(bargap=BAR_GAP)
    figure.update_yaxes(autorange="reversed", showgrid=False)
    figure.update_xaxes(title=readable(spec["x"]),
                        tickformat=tick_format(data[spec["x"]]))
    height = 110 + BAR_SLOT_PX * len(data)
    return base_layout(figure, theme, spec["title"], height)


def draw_grouped_bar(frame, spec, theme):
    colors = series_colors(frame[spec["color"]], theme)
    figure = go.Figure()

    for name, color in colors.items():
        part = frame[frame[spec["color"]].astype(str) == name]
        figure.add_trace(go.Bar(
            x=part[spec["x"]], y=part[spec["y"]].astype(str), name=name,
            orientation="h", marker={"color": color, "line": {"width": 0}},
            hovertemplate=f"{name} · %{{y}}: %{{x:,}}<extra></extra>",
        ))

    # bargroupgap leaves a visible surface gap between bars of a group.
    figure.update_layout(barmode="group", bargap=0.3, bargroupgap=0.2)
    figure.update_yaxes(autorange="reversed", showgrid=False)
    figure.update_xaxes(title=readable(spec["x"]),
                        tickformat=tick_format(frame[spec["x"]]))
    height = 130 + BAR_SLOT_PX * frame[spec["y"]].nunique() * max(1, len(colors) // 2)
    return base_layout(figure, theme, spec["title"], min(height, 900))


def line_trace(x, y, color, theme, name=None, show_end_label=True):
    """2 px line, 8 px markers with a surface ring, value at the end."""

    text = None

    if show_end_label and y.notna().any():
        last = y.last_valid_index()
        text = [format_number(value) if index == last else ""
                for index, value in y.items()]

    return go.Scatter(
        x=x, y=y, name=name, connectgaps=False,
        mode="lines+markers+text" if show_end_label else "lines+markers",
        line={"color": color, "width": 2, "shape": "linear"},
        marker={"size": 8, "color": color,
                "line": {"width": 2, "color": theme["surface"]}},
        text=text, textposition="middle right",
        textfont={"color": theme["ink_secondary"]}, cliponaxis=False,
        hovertemplate=(f"{name} · " if name else "") + "%{x}: %{y:,}<extra></extra>",
    )


def draw_line(frame, spec, theme):
    data = frame.sort_values(spec["x"])
    figure = go.Figure()

    if spec["color"]:
        colors = series_colors(data[spec["color"]], theme)
        all_x = sorted(data[spec["x"]].astype(str).unique())

        for name, color in colors.items():
            # Every series on every x; a missing period becomes a gap in
            # the line instead of a straight segment across it.
            part = (data[data[spec["color"]].astype(str) == name]
                    .assign(**{spec["x"]: lambda d: d[spec["x"]].astype(str)})
                    .set_index(spec["x"])[spec["y"]]
                    .reindex(all_x))
            figure.add_trace(line_trace(pd.Series(all_x), part.reset_index(drop=True),
                                        color, theme, name,
                                        show_end_label=len(colors) <= 4))
    else:
        figure.add_trace(line_trace(data[spec["x"]].astype(str),
                                    data[spec["y"]], theme["series"][0], theme))
        figure.update_layout(showlegend=False)

    figure.update_layout(hovermode="x unified")
    figure.update_xaxes(type="category", showgrid=False, showspikes=True,
                        spikecolor=theme["axis"], spikethickness=1,
                        spikedash="solid")
    figure.update_yaxes(title=readable(spec["y"]),
                        tickformat=tick_format(data[spec["y"]]))
    return base_layout(figure, theme, spec["title"], 380)


def draw_multiples(frame, spec, theme):
    """One panel per measure, each with its own y-axis (never overlaid)."""

    data = frame.sort_values(spec["x"])
    measures = spec["y"]
    figure = make_subplots(rows=len(measures), cols=1, shared_xaxes=True,
                           vertical_spacing=0.08,
                           subplot_titles=[readable(m) for m in measures])

    for row, measure in enumerate(measures, start=1):
        figure.add_trace(line_trace(data[spec["x"]].astype(str), data[measure],
                                    theme["series"][0], theme, readable(measure)),
                         row=row, col=1)
        figure.update_yaxes(tickformat=tick_format(data[measure]), row=row, col=1)

    figure.update_layout(showlegend=False, hovermode="x unified")
    figure.update_xaxes(type="category", showgrid=False)
    figure.update_annotations(font={"color": theme["ink_secondary"], "size": 12},
                              x=0, xanchor="left")
    return base_layout(figure, theme, spec["title"], 160 + 200 * len(measures))


def draw_scatter(frame, spec, theme):
    figure = go.Figure(go.Scatter(
        x=frame[spec["x"]], y=frame[spec["y"]], mode="markers",
        marker={"size": 9, "color": theme["series"][0],
                "line": {"width": 2, "color": theme["surface"]}},
        hovertemplate=f"{readable(spec['x'])}: %{{x:,}}<br>"
                      f"{readable(spec['y'])}: %{{y:,}}<extra></extra>",
    ))
    figure.update_xaxes(title=readable(spec["x"]))
    figure.update_yaxes(title=readable(spec["y"]))
    return base_layout(figure, theme, spec["title"], 420)


def draw_histogram(frame, spec, theme):
    figure = go.Figure(go.Histogram(
        x=frame[spec["x"]], marker={"color": theme["series"][0],
                                    "line": {"width": 0}},
        hovertemplate="%{x}: %{y} rows<extra></extra>",
    ))
    figure.update_layout(bargap=0.06)
    figure.update_xaxes(title=readable(spec["x"]), showgrid=False)
    figure.update_yaxes(title="Rows")
    return base_layout(figure, theme, spec["title"], 380)


DRAWERS = {"stat": draw_stat, "bar": draw_bar, "grouped_bar": draw_grouped_bar,
           "line": draw_line, "multiples": draw_multiples,
           "scatter": draw_scatter, "histogram": draw_histogram}


def build_figure(frame, spec, mode="light"):
    """Plotly figure for a spec, or None for table / none."""

    drawer = DRAWERS.get(spec["type"])
    return drawer(frame, spec, THEMES[mode]) if drawer else None


sample = pd.DataFrame({"city": ["Delhi", "Pune", "Mumbai"], "orders": [14, 2, 4]})
build_figure(sample, choose_chart(sample, "Orders by city?"))

## 6. Caveats from the cleaning flags

Added by code, never by the model: for every column the SQL mentions,
the warnings notebook 01 recorded (missing values, extreme values,
similar category spellings, text that could not be converted).

In [6]:
# ============================================================
# 6. CAVEATS FROM THE CLEANING FLAGS
# ============================================================

def load_profile(dataset_name):
    with (DATASETS_DIR / dataset_name / "profile.json").open(encoding="utf-8") as file:
        return json.load(file)


def caveat_text(flag):
    column, issue, detail = flag["column"], flag["issue"], flag["detail"]

    if issue == "missing_values":
        count = detail.split(" of ")[0]
        return (f"{readable(column)} has {count} missing value(s); they are "
                "left out of sums, averages and counts of that column.")

    if issue == "outliers":
        examples = re.search(r"\[(.*?)\]", detail)
        shown = examples.group(1) if examples else "one value"
        return (f"{readable(column)} contains an extreme value ({shown}) that "
                "strongly affects totals and averages.")

    if issue == "similar_categories":
        pairs = re.search(r"\[(.*)\]", detail)
        return (f"{readable(column)} may hold the same category under two "
                f"spellings: {pairs.group(1) if pairs else detail}.")

    if issue == "partly_numeric":
        return f"{readable(column)} is stored as text because some values are not numbers."

    if issue == "ambiguous_dates":
        return f"{readable(column)} dates could be day-first or month-first, so they are kept as text."

    return None


def build_caveats(sql, profile, truncated=False):
    """Warnings for the columns this query uses."""

    caveats = []

    for table in profile["tables"].values():
        if not re.search(rf"\b{re.escape(table['table_name'])}\b", sql):
            continue

        for flag in table["flags"]:
            if flag["column"] and re.search(rf"\b{re.escape(flag['column'])}\b", sql):
                text = caveat_text(flag)

                if text and text not in caveats:
                    caveats.append(text)

    if truncated:
        caveats.append("Only the first 1,000 rows are shown.")

    return caveats


sales_profile = load_profile("messy_sales")
build_caveats("SELECT city, SUM(total_sales) FROM messy_sales GROUP BY city",
              sales_profile)

['Total sales contains an extreme value (9999999) that strongly affects totals and averages.',
 "City may hold the same category under two spellings: ('Mumbai', 'Mumbay')."]

## 7. Summary with a number check

The model sees the question, the SQL and up to 20 result rows, and must
copy numbers exactly. Every number in its summary is then looked up in
the result (cell values, numbers inside text cells, the row count) and
the question. Formatting is tolerated (`805,358.86` for `805358.857`,
`25%` for `0.25`); invented or misread numbers are not. One retry with
the bad numbers listed, then a fixed template.

In [7]:
# ============================================================
# 7. SUMMARY WITH A NUMBER CHECK
# ============================================================

SUMMARY_PROMPT = """You explain database query results to a non-technical reader.

Write one to three short sentences that directly answer the question.
- Use only numbers that appear in the result table. You may add thousands separators and round decimals to 2 places.
- Do not abbreviate (no "k" or "million"), add up, subtract or otherwise calculate new numbers.
- Do not add currency symbols or units ($, Rs, kg); the data does not say which apply.
- Do not guess causes or mention SQL, tables or columns by their technical names.
- If the result has no rows, say that no data matched the question."""

SUMMARY_SCHEMA = {
    "type": "object",
    "properties": {"summary": {"type": "string"}},
    "required": ["summary"],
}


def numbers_in_text(text):
    """'Delhi had 1,234.5 sales' -> [1234.5]"""

    return [float(match.replace(",", ""))
            for match in NUMBER_PATTERN.findall(str(text))]


def allowed_numbers(frame, question):
    """Every number the summary may legitimately mention."""

    allowed = set(ALWAYS_ALLOWED_NUMBERS) | {len(frame)}
    allowed.update(numbers_in_text(question))

    for column in frame.columns:
        for value in frame[column].dropna():
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                allowed.add(float(value))
            else:
                allowed.update(numbers_in_text(value))

    return allowed


def number_is_allowed(number, allowed):
    for value in allowed:
        for candidate in (value, value * 100):           # 0.25 -> 25%
            if math.isclose(number, candidate, rel_tol=0.005, abs_tol=0.01):
                return True
    return False


def unverified_numbers(summary, frame, question):
    allowed = allowed_numbers(frame, question)
    return [number for number in numbers_in_text(summary)
            if not number_is_allowed(abs(number), {abs(v) for v in allowed})]


def result_as_text(frame):
    shown = frame.head(SUMMARY_MAX_ROWS)
    text = shown.to_csv(index=False)

    if len(frame) > len(shown):
        text += f"... ({len(frame)} rows in total)\n"

    return text


def template_summary(frame, spec):
    """Fallback that cannot invent anything."""

    if frame.empty:
        return "No data matched the question."

    if spec["type"] == "stat":
        return f"{readable(spec['y'])}: {format_number(frame[spec['y']].iloc[0])}."

    if spec["type"] == "bar":
        top = frame.sort_values(spec["x"], ascending=False).iloc[0]
        return (f"The result has {len(frame)} rows. The highest "
                f"{readable(spec['x']).lower()} is {top[spec['y']]} with "
                f"{format_number(top[spec['x']])}.")

    return f"The result has {len(frame)} rows; see the table and chart."


def summarize(settings, question, sql, frame, spec):
    """
    Returns:
        {"summary", "verified", "source", "attempts", "rejected_numbers"}
        source: model | template
    """

    if frame.empty:
        return {"summary": template_summary(frame, spec), "verified": True,
                "source": "template", "attempts": 0, "rejected_numbers": []}

    messages = [
        {"role": "system", "content": SUMMARY_PROMPT},
        {"role": "user", "content": f"Question: {question}\n\nSQL: {sql}\n\n"
                                    f"Result:\n{result_as_text(frame)}"},
    ]
    rejected = []

    for attempt in (1, 2):
        try:
            answer, _ = chat_json(settings, messages, SUMMARY_SCHEMA)

        except (ValueError, httpx.HTTPError, ollama.ResponseError):
            break                        # fall back to the template below

        summary = answer["summary"].strip()
        bad = unverified_numbers(summary, frame, question)

        if not bad:
            return {"summary": summary, "verified": True, "source": "model",
                    "attempts": attempt, "rejected_numbers": rejected}

        rejected.extend(bad)
        messages += [
            {"role": "assistant", "content": json.dumps(answer)},
            {"role": "user", "content":
                f"These numbers are not in the result: {bad}. Rewrite the "
                "summary using only numbers copied from the result."},
        ]

    return {"summary": template_summary(frame, spec), "verified": True,
            "source": "template", "attempts": 2, "rejected_numbers": rejected}


frame = run_query(database_path("messy_sales"),
                  "SELECT city, SUM(total_sales) AS total_sales FROM messy_sales "
                  "GROUP BY city ORDER BY total_sales DESC")
spec = choose_chart(frame, "What are total sales by city?")
summarize(model_settings, "What are total sales by city?",
          "SELECT city, SUM(total_sales) ...", frame, spec)

{'summary': 'Total sales by city are: Delhi with 11,275,024, Bengaluru with 585,250, Mumbai with 506,600, New Delhi with 366,830, Pune with 259,500, and Mumbay with 127,800.',
 'verified': True,
 'source': 'model',
 'attempts': 1,
 'rejected_numbers': []}

## 8. Every dev question end to end

The SQL comes from notebook 05's run (7B model). Each answer gets a
chart, a checked summary and caveats.

In [8]:
# ============================================================
# 8. EVERY DEV QUESTION END TO END
# ============================================================

def present_answer(settings, dataset_name, question, sql):
    """Result -> chart spec, figures (light + dark), summary, caveats."""

    frame = run_query(database_path(dataset_name), sql)
    spec = choose_chart(frame, question)
    summary = summarize(settings, question, sql, frame, spec)

    return {
        "question": question, "sql": sql, "frame": frame, "spec": spec,
        "figure": build_figure(frame, spec, "light"),
        "figure_dark": build_figure(frame, spec, "dark"),
        "summary": summary,
        "caveats": build_caveats(sql, load_profile(dataset_name)),
    }


dev_results = pd.read_csv(DEV_RESULTS_PATH)
dev_answers = dev_results[(dev_results["model"] == model_settings["model"])
                          & (dev_results["status"] == "answered")]
datasets = {item["question"]: item["dataset"] for item in json.load(
    (BASE_DIR / "data" / "input_data" / "dev_questions.json").open(encoding="utf-8"))}

presented = []

for row in dev_answers.itertuples():
    started = time.perf_counter()
    answer = present_answer(model_settings, datasets[row.question],
                            row.question, row.sql)
    answer["seconds"] = round(time.perf_counter() - started, 2)
    presented.append(answer)
    logger.info("%-11s %-8s %s", answer["spec"]["type"],
                answer["summary"]["source"], row.question)

overview = pd.DataFrame([{
    "question": a["question"], "chart": a["spec"]["type"],
    "summary": a["summary"]["summary"], "source": a["summary"]["source"],
    "summary_attempts": a["summary"]["attempts"],
    "rejected_numbers": a["summary"]["rejected_numbers"],
    "caveats": len(a["caveats"]), "seconds": a["seconds"],
} for a in presented])

pd.set_option("display.max_colwidth", 110)
overview

2026-11-14 18:18:00,863 | INFO | stat        model    How many orders are there?


2026-11-14 18:18:03,924 | INFO | stat        model    How many orders were placed in Delhi?


2026-11-14 18:18:10,680 | INFO | bar         template What are the total sales by region?


2026-11-14 18:18:13,701 | INFO | stat        model    Which product sold the most units?


2026-11-14 18:18:17,331 | INFO | line        model    Show total sales for each month.


2026-11-14 18:18:20,418 | INFO | stat        model    What is the average discount on paid orders?


... (6 outputs left out)

2026-11-14 18:18:45,957 | INFO | stat        model    How many active employees work in Sales?


2026-11-14 18:18:48,956 | INFO | stat        model    How many employees joined after 2020?


2026-11-14 18:18:52,188 | INFO | stat        model    What is the Q2 target for the Finance department?


,question,chart,summary,source,summary_attempts,rejected_numbers,caveats,seconds
0,How many orders are there?,stat,There are 32 orders.,model,1,[],0,2.90
1,How many orders were placed in Delhi?,stat,14 orders were placed in Delhi.,model,1,[],1,3.06
2,What are the total sales by region?,bar,"The result has 3 rows. The highest total sales is North with 11,641,854.",template,2,"[5852.0, 50.0, 5852.0, 50.0]",1,6.76
3,Which product sold the most units?,stat,Monitor sold 52 units.,model,1,[],1,3.02
4,Show total sales for each month.,line,"In January 2024, total sales were 11,593,174. In February 2024, total sales were 981,180. In March 2024, t...",model,1,[],1,3.63
5,What is the average discount on paid orders?,stat,The average discount on paid orders is 5.29.,model,1,[],0,3.09
6,Who are the top 3 customers by total sales?,bar,"The result has 3 rows. The highest total sales is Rohan Das with 10,212,499.",template,2,"[12004.0, 5206.0, 12004.0, 5206.0]",1,6.73
7,How many orders have not been paid?,stat,There are 15 orders that have not been paid.,model,1,[],0,2.97
8,What is the median total sales for each product?,bar,"The median total sales for Headphones is 16,625.00, Keyboard is 9,450.00, Laptop is 418,000.00, Monitor is...",model,1,[],1,3.59
9,Which sales rep handled the most orders?,stat,Meera handled 11 orders.,model,1,[],0,3.04


In [9]:
# A few of the charts (all of them are in charts/gallery.html).
for answer in presented:
    if answer["spec"]["type"] in {"bar", "line", "stat"}:
        print(answer["summary"]["summary"])
        for caveat in answer["caveats"]:
            print("  Note:", caveat)
        answer["figure"].show()

There are 32 orders.


14 orders were placed in Delhi.
  Note: City may hold the same category under two spellings: ('Mumbai', 'Mumbay').


The result has 3 rows. The highest total sales is North with 11,641,854.
  Note: Total sales contains an extreme value (9999999) that strongly affects totals and averages.


Monitor sold 52 units.
  Note: Units sold has 3 missing value(s); they are left out of sums, averages and counts of that column.


In January 2024, total sales were 11,593,174. In February 2024, total sales were 981,180. In March 2024, total sales were 546,650.
  Note: Total sales contains an extreme value (9999999) that strongly affects totals and averages.


The average discount on paid orders is 5.29.


... (5 outputs left out)

3 orders are missing the number of units sold.
  Note: Units sold has 3 missing value(s); they are left out of sums, averages and counts of that column.


3 active employees work in Sales.


15 employees joined after 2020.


The Q2 target for the Finance department is 120,000.


## 9. Checks

In [10]:
# ============================================================
# 9. CHECKS
# ============================================================

# ---- chart choice -------------------------------------------
pick = lambda frame: choose_chart(frame)["type"]
assert pick(pd.DataFrame()) == "none"
assert pick(pd.DataFrame({"orders": [32]})) == "stat"
assert pick(pd.DataFrame({"month": ["2024-01", "2024-02"], "s": [1, 2]})) == "line"
assert pick(pd.DataFrame({"month": ["2024-01", "2024-02"], "a": [1, 2],
                          "b": [1000, 2000]})) == "multiples"
assert pick(pd.DataFrame({"city": ["A", "B"], "s": [1, 2]})) == "bar"
assert pick(pd.DataFrame({"r": ["N", "S"], "p": ["y", "n"], "s": [1, 2]})) == "grouped_bar"
assert pick(pd.DataFrame({"r": list("ab"), "p": list("cd"), "s": [1, 2],
                          "x": [1, 2]}).assign(p=[str(i) for i in range(2)])) == "grouped_bar"
assert pick(pd.DataFrame({"a": range(6), "b": range(6)})) == "scatter"
assert pick(pd.DataFrame({"a": range(20)})) == "histogram"
assert pick(pd.DataFrame({"name": ["x", "y"]})) == "table"
assert pick(pd.DataFrame({"year": [2023, 2024], "n": [1, 2]})) == "line"
assert pick(pd.DataFrame({"code": [2023, 2024], "n": [1, 2]})) == "scatter" \
    or pick(pd.DataFrame({"code": [2023, 2024], "n": [1, 2]})) == "table"
many = pd.DataFrame({"c": [f"c{i}" for i in range(40)], "n": range(40)})
assert choose_chart(many)["note"] == f"Chart shows the top {MAX_BARS} of 40 rows."

# ---- figures ------------------------------------------------
for answer in presented:
    for figure, mode in ((answer["figure"], "light"), (answer["figure_dark"], "dark")):
        if figure is None:
            continue
        layout = figure.to_dict()["layout"]
        assert layout["paper_bgcolor"] == THEMES[mode]["surface"]
        # Never two y-scales on one plot.
        assert not any(axis.get("overlaying") for key, axis in layout.items()
                       if key.startswith("yaxis") and isinstance(axis, dict))

bar = build_figure(many, choose_chart(many))
assert len(bar.data[0].y) == MAX_BARS and bar.data[0].x[0] == 39   # largest first
assert bar.layout.height == 110 + BAR_SLOT_PX * MAX_BARS

gappy = pd.DataFrame({"month": ["2024-01", "2024-03", "2024-01", "2024-02"],
                      "region": ["West", "West", "North", "North"], "n": [5, 3, 1, 2]})
lines = build_figure(gappy, choose_chart(gappy))
west = next(trace for trace in lines.data if trace.name == "West")
assert list(west.x) == ["2024-01", "2024-02", "2024-03"] and pd.isna(west.y[1])

tile = build_figure(pd.DataFrame({"product": ["Monitor"], "units": [52]}),
                    {"type": "stat", "y": "units", "title": "t"})
assert tile.data[0].title.text == "Monitor · Units"

colors_a = series_colors(["Pune", "Delhi"], THEMES["light"])
colors_b = series_colors(["Delhi"], THEMES["light"])
assert colors_a["Delhi"] == colors_b["Delhi"]          # colour follows the name

# ---- numbers ------------------------------------------------
result = pd.DataFrame({"city": ["Delhi"], "total": [805358.857], "share": [0.25],
                       "month": ["2024-01"]})
assert numbers_in_text("Delhi had 1,234.5 sales in 2024") == [1234.5, 2024.0]
assert unverified_numbers("Delhi sold 805,358.86 (25%) in 2024.", result, "q") == []
assert unverified_numbers("Delhi sold 805,359 units.", result, "q") == []    # rounding
assert unverified_numbers("Delhi sold 900,000.", result, "q") == [900000.0]
assert unverified_numbers("The top 3 cities", result, "top 3 cities") == []

# ---- caveats ------------------------------------------------
caveats = build_caveats("SELECT SUM(total_sales) FROM messy_sales", sales_profile)
assert any("9999999" in text for text in caveats)
assert build_caveats("SELECT COUNT(*) FROM messy_sales", sales_profile) == []
assert any("Mumbay" in text for text in
           build_caveats("SELECT city FROM messy_sales", sales_profile))

# ---- end to end ---------------------------------------------
assert all(answer["summary"]["verified"] for answer in presented)
assert not overview["summary"].str.contains(r"[$€£₹]").any(), "currency symbol added"
assert overview["chart"].isin(set(DRAWERS) | {"table"}).all()

print("All checks passed.")
print(overview["chart"].value_counts().to_dict(),
      "| summaries from model:", int((overview["source"] == "model").sum()),
      "of", len(overview))

All checks passed.


{'stat': 11, 'bar': 3, 'line': 1} | summaries from model: 13 of 15


## 10. Save a gallery

All charts with their summaries and caveats in one HTML page, in light
and dark, to review by eye.

In [11]:
# ============================================================
# 10. SAVE A GALLERY
# ============================================================

def gallery_html(answers, mode):
    theme = THEMES[mode]
    parts = [f"<html><head><meta charset='utf-8'><title>Charts ({mode})</title>"
             f"<style>body{{background:{theme['surface']};color:{theme['ink']};"
             f"font-family:{FONT_FAMILY};max-width:820px;margin:24px auto;padding:0 16px}}"
             f".note{{color:{theme['ink_secondary']};font-size:13px}}"
             f"section{{border-top:1px solid {theme['grid']};padding:16px 0}}</style>"
             "</head><body>"]
    first = True

    for answer in answers:
        figure = answer["figure" if mode == "light" else "figure_dark"]
        parts.append(f"<section><h3>{answer['question']}</h3>"
                     f"<p>{answer['summary']['summary']}</p>")
        parts += [f"<p class='note'>Note: {text}</p>" for text in answer["caveats"]]
        parts.append(f"<p class='note'><code>{answer['sql']}</code></p>")

        if figure is not None:
            parts.append(figure.to_html(full_html=False,
                                        include_plotlyjs="cdn" if first else False))
            first = False
        else:
            parts.append(answer["frame"].head(10).to_html(index=False))

        parts.append("</section>")

    return "".join(parts) + "</body></html>"


for mode in ("light", "dark"):
    (OUTPUT_DIR / f"gallery_{mode}.html").write_text(gallery_html(presented, mode),
                                                     encoding="utf-8")

overview.to_csv(OUTPUT_DIR / "summaries.csv", index=False)
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

['gallery_dark.html', 'gallery_light.html', 'summaries.csv']
